# Y sensitivity vs. X setpoint

Analyzes a `piezo_xy_cross_scan.py` session: the X piezo is parked at each of
several DC setpoints spanning the full range, and at each one the Y piezo is
swept full-range (a dense triangle). Each Y sub-scan CSV lives in
`scans/cross_<timestamp>/` and encodes the held X voltage in its filename as an
`_xset<V>` token.

This notebook:
1. loads every Y sub-scan of the most recent cross run (override `CROSS_DIR`),
   parsing the held X setpoint from each filename;
2. fits the **Y sensitivity (mirror urad/V)** as the slope of Y mirror angle vs.
   Y readback voltage for each sub-scan; and
3. plots **Y sensitivity vs. X voltage** (the headline result).

Same conventions as `analysis.ipynb`: displacement is the Gaussian-fit center
(not the raw centroid), converted to a mirror tilt angle via `DETECTOR_DISTANCE`
(a mirror tilt θ steers the beam by 2θ), and the camera is mounted rotated 90 deg
so the scanned piezo Y axis lands on **camera X** (`CAM_OF_PIEZO['Y'] = 'X'`).
`OMIT_FIRST_PASS` drops each scan's initial warm-up up-ramp.

In [ ]:
import glob
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

DATA_DIR = "scans"

# Distance from the steering mirror to the beam-profiler sensor (meters), used to
# convert a camera-plane beam displacement into the mirror tilt angle that caused
# it. Keep in sync with analysis.ipynb.
DETECTOR_DISTANCE = 0.42

# The camera is rotated 90 deg: piezo X -> camera Y, piezo Y -> camera X. Here Y
# is the scanned axis, so its motion is read from camera X.
CAM_OF_PIEZO = {"X": "Y", "Y": "X", "Z": "Y"}

# Drop each scan's first up-ramp (warm-up/conditioning), same toggle as analysis.ipynb.
OMIT_FIRST_PASS = True


def um_to_mirror_urad(disp_um):
    """Camera-plane beam displacement (um) -> mirror tilt angle (microradians).
    A mirror tilted by theta steers the beam by 2*theta; over distance L the beam
    lands d = 2*theta*L off-center, so theta = d / (2 L). With d in um and L in m
    the unit factors cancel, giving microradians directly."""
    return disp_um / (2.0 * DETECTOR_DISTANCE)


def add_mirror_angle_cols(d):
    """Add a `_urad` mirror-angle column beside every `_um` displacement column."""
    for col in [c for c in d.columns if c.endswith("_um")]:
        d[col[:-len("_um")] + "_urad"] = um_to_mirror_urad(d[col])
    return d


def drop_first_upramp(d):
    """Return `d` with the initial rising ramp removed (honours OMIT_FIRST_PASS).
    Uses the `points_per_ramp` column when present, else infers the ramp length
    from the first peak of the voltage-setpoint triangle."""
    if not OMIT_FIRST_PASS or len(d) < 2:
        return d
    if "points_per_ramp" in d.columns and d["points_per_ramp"].notna().all():
        ppr = int(d["points_per_ramp"].iloc[0])
    else:
        ppr = int(np.argmax(d["voltage_setpoint_V"].to_numpy()))
    if ppr <= 0 or ppr >= len(d):
        return d
    return d.iloc[ppr:].reset_index(drop=True)

In [ ]:
# --- select a cross run and load its Y sub-scans ---
cross_dirs = sorted(glob.glob(os.path.join(DATA_DIR, "cross_*")))
CROSS_DIR = cross_dirs[-1] if cross_dirs else None
# CROSS_DIR = "scans/cross_20260721_120000"   # override to pick a specific run
print("Cross runs:", [os.path.basename(d) for d in cross_dirs])
print("Using:", CROSS_DIR)

# filename: piezo_scan_<stamp>_min<..>_max<..>_axisY_xset<Vheld>.csv
_SUB_RE = re.compile(r"axis([XYZ])_([xyz])set([\d.]+)", re.IGNORECASE)


def load_subscan(path):
    """Return (meta, dataframe) for one Y sub-scan. meta carries the scanned axis,
    the held axis + its parked voltage, and pos_col -- the mirror-angle (urad)
    column of the camera axis that carries the scanned axis's motion."""
    m = _SUB_RE.search(os.path.basename(path))
    scan_axis = m.group(1).upper() if m else "Y"
    held_axis = m.group(2).upper() if m else "X"
    held_v = float(m.group(3)) if m else np.nan
    d = pd.read_csv(path, parse_dates=["timestamp"])
    d = add_mirror_angle_cols(d)
    d = drop_first_upramp(d)
    meta = {
        "path": path,
        "scan_axis": scan_axis,
        "held_axis": held_axis,
        "held_v": held_v,
        "pos_col": f"gaussfit_center_{CAM_OF_PIEZO[scan_axis].lower()}_urad",
    }
    return meta, d


subscan_files = sorted(glob.glob(os.path.join(CROSS_DIR, "*.csv"))) if CROSS_DIR else []
print(f"{len(subscan_files)} Y sub-scans found")

records = []
subscans = []   # list of (meta, df), reused by the plots below
for path in subscan_files:
    meta, d = load_subscan(path)
    pos = d[meta["pos_col"]]
    # Y sensitivity: slope of mirror angle vs. readback voltage (urad/V)
    slope, _ = np.polyfit(d["voltage_readback_V"], pos, 1)
    # hysteresis: max spread in mirror angle among samples at the same setpoint
    spread = d.groupby("voltage_setpoint_V")[meta["pos_col"]].agg(lambda s: s.max() - s.min())
    records.append({
        "held_axis": meta["held_axis"],
        "x_setpoint_V": meta["held_v"],
        "y_sensitivity_urad_per_V": slope,
        "hysteresis_urad": spread.max(),
    })
    subscans.append((meta, d))

summary = (pd.DataFrame(records)
           .sort_values("x_setpoint_V")
           .reset_index(drop=True)) if records else pd.DataFrame(
    columns=["held_axis", "x_setpoint_V", "y_sensitivity_urad_per_V", "hysteresis_urad"])
summary

In [ ]:
# Headline result: Y sensitivity (mirror angle per volt) as a function of where
# the X axis is parked. A flat line means Y sensitivity is independent of X
# position; a slope would mean the two axes' drives are coupled.
if not summary.empty:
    fig, (ax_s, ax_h) = plt.subplots(1, 2, figsize=(12, 5))
    ax_s.plot(summary["x_setpoint_V"], summary["y_sensitivity_urad_per_V"], "o-")
    ax_s.set_xlabel("X setpoint (V, held)")
    ax_s.set_ylabel("Y sensitivity (urad/V)")
    ax_s.set_title("Y sensitivity vs. X voltage")

    ax_h.plot(summary["x_setpoint_V"], summary["hysteresis_urad"], "o-", color="tab:orange")
    ax_h.set_xlabel("X setpoint (V, held)")
    ax_h.set_ylabel("Y max hysteresis (urad)")
    ax_h.set_title("Y hysteresis vs. X voltage")

    fig.suptitle(f"Cross run ({os.path.basename(CROSS_DIR)})")
    fig.tight_layout()
    plt.show()

    span = summary["y_sensitivity_urad_per_V"].max() - summary["y_sensitivity_urad_per_V"].min()
    mean = summary["y_sensitivity_urad_per_V"].mean()
    print(f"Y sensitivity: mean {mean:.3f} urad/V, "
          f"spread {span:.3f} urad/V ({100 * span / mean:.1f}% of mean) across X setpoints.")
else:
    print("No sub-scans to plot.")

In [ ]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

# Per-subscan Y transfer curve: one subplot per held-X setpoint, raw detector
# displacement (um) vs. Y voltage, colored by scan progress so the rising/falling
# legs (any hysteresis loop) are visible. Sanity-check the fits above.
if subscans:
    n = len(subscans)
    ncols = min(4, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.6 * nrows),
                             squeeze=False,
                             gridspec_kw={"hspace": 0.55, "wspace": 0.4})
    axes_flat = axes.ravel()
    norm = Normalize(0, 1)
    for ax, (meta, d) in zip(axes_flat, sorted(subscans, key=lambda md: md[0]["held_v"])):
        um_col = meta["pos_col"].replace("_urad", "_um")
        progress = np.linspace(0, 1, len(d))
        ax.scatter(d["voltage_readback_V"], d[um_col], c=progress,
                   cmap="viridis", norm=norm, s=12)
        ax.plot(d["voltage_readback_V"], d[um_col], "-", color="gray",
                alpha=0.3, linewidth=0.7)
        ax.set_title(f"{meta['scan_axis']} scan @ {meta['held_axis']}={meta['held_v']:.0f} V", fontsize=9)
        ax.set_xlabel("Y V readback")
        ax.set_ylabel(f"{meta['scan_axis']} disp (um)")
    for ax in axes_flat[n:]:
        ax.axis("off")

    sm = ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes_flat.tolist())
    cbar.set_label("Scan progress (start → end)")
    fig.suptitle(f"Per-subscan Y transfer curves ({os.path.basename(CROSS_DIR)})")
    plt.show()
else:
    print("No sub-scans to plot.")